In [30]:
import pandas as pd
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import os
sns.set_theme()
from sklearn.preprocessing import LabelEncoder
from sklearn import linear_model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
cwd = os.getcwd()
print("Current working directory:", cwd)

# Change the working directory
#os.chdir('S:/OneDrive - University of Georgia/1 UGA/1 PhD/1 Spring24/Tobacco')
#os.chdir(r'/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco')

os.chdir(r'/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_Chapter/madeData')

# Verify the change
print("New working directory:", os.getcwd())

Current working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_chapter/madeData
New working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/Fall25/First_chapter/madeData


Consumer Price Index

In [31]:
# Loading the dataset
df = pd.read_excel('monthly_CPI_corr.xlsx')

df # region week level

,region,CPI,W,ln_CPI
0,1,98.5,1,4.590057
1,1,98.5,2,4.590057
2,1,98.5,3,4.590057
3,1,98.5,4,4.590057
4,1,98.5,5,4.590057
...,...,...,...,...
2191,12,99.9,179,4.604170
2192,12,100.6,180,4.611152
2193,12,100.0,181,4.605170
2194,12,100.6,182,4.611152


In [32]:

# Ensure data is sorted by Region and Week
df = df.sort_values(by=['region', 'W'])

# 2. Identify "Monthly Blocks"
# We define a block as a sequence of consecutive weeks with the same CPI within a region.
# This works even without a "Time Code" column.
# 'condition' is True when CPI changes OR Region changes
condition = (df['CPI'] != df['CPI'].shift()) | (df['region'] != df['region'].shift())
df['block_id'] = condition.cumsum()

# 3. Find the "Anchor Week" for each Block
# The anchor is the median week of the block (the "middle" of the month)
# We calculate the median W for each block_id
block_medians = df.groupby('block_id')['W'].median().round().astype(int).reset_index()
block_medians.rename(columns={'W': 'W_anchor'}, inplace=True)

# Merge the anchor week back to the main dataframe
df = pd.merge(df, block_medians, on='block_id', how='left')

# 4. Set Interpolation Points
# We create a column that is NaN everywhere EXCEPT at the anchor week
df['CPI_to_interp'] = np.where(df['W'] == df['W_anchor'], df['CPI'], np.nan)

# 5. Perform Linear Interpolation
# We group by region so interpolation doesn't cross regional boundaries
df['CPI_Smooth'] = df.groupby('region')['CPI_to_interp'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

# 6. Final Cleanup
# Calculate Log CPI as well since AIDS models use logs
df['ln_CPI_Smooth'] = np.log(df['CPI_Smooth'])

# Select final columns
df_final = df[['region', 'W', 'CPI_Smooth', 'ln_CPI_Smooth']].copy()
df_final.rename(columns={'CPI_Smooth': 'CPI', 'ln_CPI_Smooth': 'ln_CPI'}, inplace=True)

# 7. Check the Results
print("Head (Region 1 - Start):")
print(df_final.head(10))

print("\nTail (Region 12 - End):")
print(df_final.tail(10))

df_CPI_interpolated = df_final.copy()


Head (Region 1 - Start):
   region   W     CPI    ln_CPI
0       1   1  98.500  4.590057
1       1   2  98.500  4.590057
2       1   3  98.500  4.590057
3       1   4  98.460  4.589650
4       1   5  98.420  4.589244
5       1   6  98.380  4.588838
6       1   7  98.340  4.588431
7       1   8  98.300  4.588024
8       1   9  98.275  4.587770
9       1  10  98.250  4.587515

Tail (Region 12 - End):
      region    W    CPI    ln_CPI
2186      12  174  100.6  4.611152
2187      12  175   99.9  4.604170
2188      12  176  100.6  4.611152
2189      12  177   99.9  4.604170
2190      12  178  100.6  4.611152
2191      12  179   99.9  4.604170
2192      12  180  100.6  4.611152
2193      12  181  100.0  4.605170
2194      12  182  100.6  4.611152
2195      12  183  100.0  4.605170


Monthly Income Interpolation

In [33]:


# Loading the dataset
df22 = pd.read_excel("final_inc_exp.xlsx")
df22

,region,Region,W,month,hh_dist,hh_num,Amount_received,income_yen,expenditure_yen,consumer_exp_yen,person_perhh,monthly_avg_income_hh,monthly_avg_expenditure_hh,person_perhh_avg,monthly_percap_income,monthly_percap_exp
0,1,Chugoku,1,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408
1,1,Chugoku,2,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408
2,1,Chugoku,3,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408
3,1,Chugoku,4,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408
4,1,Chugoku,5,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191,12,Tokai,179,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366
2192,12,Tokai,180,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366
2193,12,Tokai,181,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366
2194,12,Tokai,182,2020000707,35,55,1125657,677144,397684,261602,3.17,789828.5,306580.25,3.2950,239705.159332,93044.081942


In [34]:


# 1. Create the Grouping Key (same as before)
# This groups all weekly rows belonging to the same Region and Month.
df22['RegionMonth'] = df22['region'].astype(str) + '_' + df22['month'].astype(str)

# --- Define the Apportionment Function ---
def apportion_with_variation(series_data):
    """
    Takes a series of repeated monthly values, introduces random variation,
    and scales the results to ensure they sum back to the original value.
    """
    monthly_total = series_data.iloc[0] # The constant monthly value
    num_weeks = len(series_data)
    
    # Generate a random factor for each week (e.g., uniform distribution)
    # The minimum factor must be > 0.
    random_factors = np.random.rand(num_weeks) + 0.5 # Adds 0.5 to keep factors well above zero
    
    # Calculate the total of the random factors for scaling
    total_factor_sum = random_factors.sum()
    
    # Calculate the weekly value: (Monthly Total) * (Individual Factor / Total Factor Sum)
    # This ensures the new weekly values sum up to the Monthly Total.
    weekly_values = monthly_total * (random_factors / total_factor_sum)
    
    return weekly_values

# --- Apply the function to both income and expenditure ---

# 2. Calculate Weekly Per Capita Income with Variation
df22['weekly_percap_income'] = (
    df22.groupby('RegionMonth')['monthly_percap_income']
    .transform(apportion_with_variation)
)

# 3. Calculate Weekly Per Capita Expenditure with Variation
df22['weekly_percap_exp'] = (
    df22.groupby('RegionMonth')['monthly_percap_exp']
    .transform(apportion_with_variation)
)

# 4. Cleanup
df22.drop(columns=['RegionMonth'], inplace=True)

# Verification: Check that a sample group sums correctly (Optional)
sample_check = df22.groupby(['Region', 'month'])[['monthly_percap_income', 'weekly_percap_income']].agg({
    'monthly_percap_income': 'first', # Get the original monthly total
    'weekly_percap_income': 'sum'      # Get the sum of the new weekly totals
}).head()

print("Verification Check (Monthly Total vs. Sum of New Weekly Values):")
print(sample_check)

df22

Verification Check (Monthly Total vs. Sum of New Weekly Values):
                    monthly_percap_income  weekly_percap_income
Region  month                                                  
Chugoku 2017000101          117281.656805         117281.656805
        2017000202          130547.774481         130547.774481
        2017000303          113573.214286         113573.214286
        2017000404          128986.350148         128986.350148
        2017000505          108337.091988         108337.091988


,region,Region,W,month,hh_dist,hh_num,Amount_received,income_yen,expenditure_yen,consumer_exp_yen,person_perhh,monthly_avg_income_hh,monthly_avg_expenditure_hh,person_perhh_avg,monthly_percap_income,monthly_percap_exp,weekly_percap_income,weekly_percap_exp
0,1,Chugoku,1,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408,23411.669105,15623.322767
1,1,Chugoku,2,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408,26436.060081,13725.202346
2,1,Chugoku,3,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408,25093.736984,25808.476886
3,1,Chugoku,4,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408,29988.758497,19632.895108
4,1,Chugoku,5,2017000101,613,338,818000,396412,368551,297416,3.38,396412.0,297416.00,3.3800,117281.656805,87992.899408,12351.432137,13203.002301
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191,12,Tokai,179,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366,82057.886177,23315.283871
2192,12,Tokai,180,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366,86519.067583,22831.037989
2193,12,Tokai,181,2020000606,36,53,1861822,1287577,601437,381354,3.20,1059640.0,293283.75,3.2975,321346.474602,88941.243366,86243.628743,19145.659118
2194,12,Tokai,182,2020000707,35,55,1125657,677144,397684,261602,3.17,789828.5,306580.25,3.2950,239705.159332,93044.081942,125832.785165,32737.476607


In [35]:
df22 = df22[['region', 'W', 'weekly_percap_income', 'weekly_percap_exp']].copy()

df22

,region,W,weekly_percap_income,weekly_percap_exp
0,1,1,23411.669105,15623.322767
1,1,2,26436.060081,13725.202346
2,1,3,25093.736984,25808.476886
3,1,4,29988.758497,19632.895108
4,1,5,12351.432137,13203.002301
...,...,...,...,...
2191,12,179,82057.886177,23315.283871
2192,12,180,86519.067583,22831.037989
2193,12,181,86243.628743,19145.659118
2194,12,182,125832.785165,32737.476607


In [36]:
#df22.to_excel('weekly_exp_interpolated_FINAL.xlsx') # region-week level


In [37]:
# Merging the CPI and Income variables
## merging these two datasets on W and region variables:
merged_df1 = pd.merge(df_CPI_interpolated, df22, on=['W', 'region'], how='inner')
merged_df1

,region,W,CPI,ln_CPI,weekly_percap_income,weekly_percap_exp
0,1,1,98.50,4.590057,23411.669105,15623.322767
1,1,2,98.50,4.590057,26436.060081,13725.202346
2,1,3,98.50,4.590057,25093.736984,25808.476886
3,1,4,98.46,4.589650,29988.758497,19632.895108
4,1,5,98.42,4.589244,12351.432137,13203.002301
...,...,...,...,...,...,...
2191,12,179,99.90,4.604170,82057.886177,23315.283871
2192,12,180,100.60,4.611152,86519.067583,22831.037989
2193,12,181,100.00,4.605170,86243.628743,19145.659118
2194,12,182,100.60,4.611152,125832.785165,32737.476607


In [38]:
merged_df1.to_csv('other_var_FINAL.csv', index=False) # region-week level
